In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    BaggingClassifier,
    AdaBoostClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss
)

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.model_selection import GridSearchCV

In [3]:
df = pd.read_csv(
    "../feature-engineering/output/dataset_churn.csv"
)

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (500, 12)
['cliente_id', 'total_atendimentos', 'total_gasto', 'ticket_medio', 'dias_desde_ultimo_atendimento', 'intervalo_medio_atendimentos', 'atendimentos_30d', 'atendimentos_90d', 'gasto_90d', 'quantidade_servicos_diferentes', 'quantidade_profissionais_diferentes', 'target_churn']


In [4]:
features_modelo_a = [
    "total_atendimentos",
    "total_gasto",
    "ticket_medio",
    "dias_desde_ultimo_atendimento",
    "intervalo_medio_atendimentos",
    "atendimentos_30d",
    "atendimentos_90d",
    "gasto_90d"
]

In [5]:
X = df[features_modelo_a]
y = df["target_churn"]

print("X:", X.shape)
print("y:", y.shape)

print("\nDistribuição do target:")
print(y.value_counts())

X: (500, 8)
y: (500,)

Distribuição do target:
target_churn
0    326
1    174
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (400, 8)
Teste: (100, 8)


In [7]:
modelos = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Bagging": BaggingClassifier(
        n_estimators=100,
        bootstrap=True
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "HistGradient Boosting": HistGradientBoostingClassifier(
        random_state=42
    ),

    "Ada Boosting": AdaBoostClassifier(
        DecisionTreeClassifier(max_depth=1),
        n_estimators=100,
        
    )
}

In [8]:
resultados = []

for nome, modelo in modelos.items():

    modelo.fit(X_train, y_train)

    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]

    resultados.append({
        "Modelo": nome,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Brier": brier_score_loss(y_test, y_prob)
    })

In [10]:
resultados_modelos = pd.DataFrame(resultados)

resultados_modelos = resultados_modelos.sort_values(
    by="ROC-AUC",
    ascending=False
)

resultados_modelos

,Modelo,Accuracy,Precision,Recall,F1,ROC-AUC,Brier
3,Random Forest,0.72,0.705882,0.342857,0.461538,0.708132,0.201551
2,Bagging,0.75,0.812500,0.371429,0.509804,0.698022,0.201584
1,Decision Tree,0.72,0.666667,0.400000,0.500000,0.675604,0.248000
5,HistGradient Boosting,0.69,0.600000,0.342857,0.436364,0.666374,0.214388
6,Ada Boosting,0.69,0.750000,0.171429,0.279070,0.637582,0.225866
4,Gradient Boosting,0.68,0.588235,0.285714,0.384615,0.632967,0.220663
0,Logistic Regression,0.69,0.750000,0.171429,0.279070,0.602198,0.215683


In [12]:
print("=== RANKING POR ROC-AUC ===")

ranking_auc = resultados_modelos[
    ["Modelo", "ROC-AUC"]
].sort_values(
    by="ROC-AUC",
    ascending=False
)

ranking_auc

=== RANKING POR ROC-AUC ===


,Modelo,ROC-AUC
3,Random Forest,0.708132
2,Bagging,0.698022
1,Decision Tree,0.675604
5,HistGradient Boosting,0.666374
6,Ada Boosting,0.637582
4,Gradient Boosting,0.632967
0,Logistic Regression,0.602198


In [13]:
print("=== RANKING POR F1 ===")

ranking_f1 = resultados_modelos[
    ["Modelo", "F1"]
].sort_values(
    by="F1",
    ascending=False
)

ranking_f1

=== RANKING POR F1 ===


,Modelo,F1
2,Bagging,0.509804
1,Decision Tree,0.500000
3,Random Forest,0.461538
5,HistGradient Boosting,0.436364
4,Gradient Boosting,0.384615
6,Ada Boosting,0.279070
0,Logistic Regression,0.279070


In [14]:
print("=== RANKING POR BRIER ===")

ranking_brier = resultados_modelos[
    ["Modelo", "Brier"]
].sort_values(
    by="Brier",
    ascending=True
)

ranking_brier

=== RANKING POR BRIER ===


,Modelo,Brier
3,Random Forest,0.201551
2,Bagging,0.201584
5,HistGradient Boosting,0.214388
0,Logistic Regression,0.215683
4,Gradient Boosting,0.220663
6,Ada Boosting,0.225866
1,Decision Tree,0.248000


# CROSS VALIDATION

In [9]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [10]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "neg_brier": "neg_brier_score"
}

In [11]:
resultados_cv = []

for nome,modelo in modelos.items():
    resultado = cross_validate(
        modelo,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    resultados_cv.append({
        "Modelo":nome,
        "Acurracy_mean": resultado["test_accuracy"].mean(),
        "Accuracy_std": resultado["test_accuracy"].std(),

        "Precision_mean": resultado["test_precision"].mean(),
        "Precision_std" : resultado["test_precision"].std(),

        "Recall_mean" : resultado["test_recall"].mean(),
        "Recal_std": resultado["test_recall"].std(),

        "F1_mean": resultado["test_f1"].mean(),
        "F1_std": resultado["test_f1"].std(),

        "ROC-AUC_mean": resultado["test_roc_auc"].mean(),
        "ROC-AUC_std" : resultado["test_roc_auc"].std(),

        "Brier_mean" : -resultado["test_neg_brier"].mean(),
        "Brier_std" : resultado["test_neg_brier"].std()
    })

In [25]:
resultados_cv = pd.DataFrame(resultados_cv)

resultados_cv = resultados_cv.sort_values(
    by="ROC-AUC_mean",
    ascending=False
)

display(resultados_cv)

,Modelo,Acurracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recal_std,F1_mean,F1_std,ROC-AUC_mean,ROC-AUC_std,Brier_mean,Brier_std
0,Logistic Regression,0.6400,0.031024,0.478904,0.168039,0.129365,0.062017,0.194895,0.076032,0.639893,0.070191,0.216298,0.017603
6,Ada Boosting,0.6350,0.048348,0.456966,0.131153,0.193651,0.056867,0.268971,0.076725,0.601548,0.035772,0.222528,0.004999
5,HistGradient Boosting,0.6000,0.041079,0.377637,0.100654,0.201323,0.027688,0.260717,0.043312,0.594103,0.030602,0.259516,0.021057
3,Random Forest,0.6225,0.034821,0.408247,0.094818,0.244180,0.108516,0.301341,0.109394,0.589285,0.034407,0.243057,0.017055
2,Bagging,0.6050,0.037583,0.379449,0.098063,0.201587,0.058504,0.259999,0.065222,0.580152,0.031427,0.249511,0.018390
4,Gradient Boosting,0.5775,0.041382,0.323429,0.082874,0.187037,0.034822,0.236677,0.050241,0.541954,0.057913,0.270621,0.030454
1,Decision Tree,0.5775,0.068191,0.362638,0.136439,0.202116,0.061222,0.249816,0.067556,0.516303,0.081832,0.379609,0.074731


# Comparação do comportamento dos modelos em relação ao objetivo de negócio e Tuning dos candidatos mais promissores.

## GridSearchCV — Random Forest

In [35]:
param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 3, 5, 7],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [36]:
rf_base = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

grid_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid_rf,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True
)

In [37]:
grid_rf.fit(
    X_train,
    y_train
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [None, 3, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0

In [38]:
print("Melhores parâmetros:")
print(grid_rf.best_params_)

print("\nMelhor ROC-AUC:")
print(grid_rf.best_score_)

Melhores parâmetros:
{'max_depth': 3, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 300}

Melhor ROC-AUC:
0.6154836297524977


In [25]:
# Logistic Regression

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

param_grid_lr = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}

grid_lr = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid_lr,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True
)

In [26]:
grid_lr.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__solver': ['liblinear', 'lbfgs']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, de

In [27]:
print("Melhores parâmetros:")
print(grid_lr.best_params_)

print("\nMelhor ROC-AUC:")
print(grid_lr.best_score_)

Melhores parâmetros:
{'model__C': 0.1, 'model__solver': 'liblinear'}

Melhor ROC-AUC:
0.6602632735887453


In [29]:
#AdaBoost

param_grid_ab = {
    "n_estimators": [50, 100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]
}

ad_base = AdaBoostClassifier(
    random_state=42
)

grid_ab = GridSearchCV(
    estimator=ad_base,
    param_grid=param_grid_ab,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True
)

In [30]:
grid_ab.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",AdaBoostClass...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.05, ...], 'n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, defau

In [31]:
print("Melhores parâmetros:")
print(grid_ab.best_params_)

print("\nMelhor ROC-AUC:")
print(grid_ab.best_score_)

Melhores parâmetros:
{'learning_rate': 0.5, 'n_estimators': 100}

Melhor ROC-AUC:
0.6216691720229457


In [39]:
# Comparar os 3 modelos masi promissores

resultado_tuning = pd.DataFrame({
    "Modelo": [
        "Random Forest",
        "Logistic Regression",
        "AdaBoost"
    ],
    "ROC-AUC_CV": [
        grid_rf.best_score_,
        grid_lr.best_score_,
        grid_ab.best_score_
    ]
})

resultados_tuning = resultado_tuning.sort_values(
    by="ROC-AUC_CV",
    ascending=False
)

resultados_tuning

,Modelo,ROC-AUC_CV
1,Logistic Regression,0.660263
2,AdaBoost,0.621669
0,Random Forest,0.615484


# Avaliação final dos três modelos tunados no X_test / y_test.

In [40]:
melhor_rf = grid_rf.best_estimator_
melhor_lr = grid_lr.best_estimator_
melhor_ab = grid_ab.best_estimator_

In [42]:
modelos = {
    "Logistic Regression": melhor_lr,
    "AdaBoost": melhor_ab,
    "Random Forest": melhor_rf
}

resultados_teste = []

for nome, modelo in modelos.items():

    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]

    resultados_teste.append({
        "Modelo": nome,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Brier": brier_score_loss(y_test, y_prob)
    })

In [43]:
df_resultados_teste = pd.DataFrame(resultados_teste)

df_resultados_teste = df_resultados_teste.sort_values(
    by="ROC-AUC",
    ascending=False
)

df_resultados_teste

,Modelo,Accuracy,Precision,Recall,F1,ROC-AUC,Brier
2,Random Forest,0.51,0.405405,0.857143,0.550459,0.662857,0.238175
1,AdaBoost,0.65,0.000000,0.000000,0.000000,0.648791,0.222907
0,Logistic Regression,0.68,0.666667,0.171429,0.272727,0.616264,0.215216


Matriz de confusão dos três

In [44]:
from sklearn.metrics import confusion_matrix


for nome, modelo in modelos.items():

    y_pred = modelo.predict(X_test)
    print("\n==============================")
    print(nome)
    print("==============================")

    print(confusion_matrix(y_test, y_pred))



Logistic Regression
[[62  3]
 [29  6]]

AdaBoost
[[65  0]
 [35  0]]

Random Forest
[[21 44]
 [ 5 30]]


In [45]:
from sklearn.metrics import classification_report


for nome, modelo in modelos.items():

    y_pred = modelo.predict(X_test)

    print("\n==============================")
    print(nome)
    print("==============================")

    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))


Logistic Regression
              precision    recall  f1-score   support

           0       0.68      0.95      0.79        65
           1       0.67      0.17      0.27        35

    accuracy                           0.68       100
   macro avg       0.67      0.56      0.53       100
weighted avg       0.68      0.68      0.61       100


AdaBoost
              precision    recall  f1-score   support

           0       0.65      1.00      0.79        65
           1       0.00      0.00      0.00        35

    accuracy                           0.65       100
   macro avg       0.33      0.50      0.39       100
weighted avg       0.42      0.65      0.51       100


Random Forest
              precision    recall  f1-score   support

           0       0.81      0.32      0.46        65
           1       0.41      0.86      0.55        35

    accuracy                           0.51       100
   macro avg       0.61      0.59      0.51       100
weighted avg       0.67     

Random Forest é o candidato mais forte para o sistema de churn, principalmente pelo Recall de 86%. Ele encvontrou miotomais clientes em churn do que os outros e para o negócio então perder um cliente em churn (FN) pode ser muito mais problemático do que abordar um cliente que talvez não estivesse em churn (FP). 